# 06 — Preference training (DPO)

Train with Direct Preference Optimization on pairs from verified execution results. Pairs whose chosen response lacks execution evidence are logged and skipped — preference labels are never manufactured from an LLM's opinion alone.

In [ ]:
import sys
sys.path.insert(0, ".")
from pathlib import Path
print("repo root:", Path.cwd())

In [ ]:
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    torch = None
    HAS_GPU = False
print("GPU available:", HAS_GPU)
if not HAS_GPU:
    print("Skipping GPU steps on this host — run this notebook on Kaggle with a GPU.")

In [ ]:
import json
from pathlib import Path
pairs_path = Path("data/dpo.jsonl")
print("preference file exists:", pairs_path.exists())
if pairs_path.exists():
    pairs = [json.loads(line) for line in pairs_path.read_text().splitlines() if line.strip()]
    verified = sum(1 for p in pairs if p.get("chosen", {}).get("execution_verified"))
    print(f"{len(pairs)} pairs, {verified} with verified execution evidence")

In [ ]:
if HAS_GPU:
    from src.training.trainer_utils import load_config
    from src.training.dpo import run_dpo
    config = load_config("configs/dpo.yaml")
    summary = run_dpo(config, config["dataset"]["train_file"], config["output_dir"])
    print("kept pairs:", summary.get("kept_pairs"), "| skipped:", summary.get("skipped_pairs"))
else:
    print("No GPU: DPO skipped. On Kaggle, re-run this cell with a GPU attached.")